In [1]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/nassau_candy_enriched.csv')
df['Order Date'] = pd.to_datetime(df['Order Date'])

print(f"Enriched data loaded: {df.shape[0]} rows, {df.shape[1]} columns")

Enriched data loaded: 6013 rows, 28 columns


### 1. Product Level Pareto (Profit)

In [3]:
total_profit  = df['Gross Profit'].sum()
total_revenue = df['Sales'].sum()

product_profit = df.groupby('Product Name')['Gross Profit'].sum()\
                   .sort_values(ascending=False)\
                   .reset_index()

product_profit.columns = ['Product Name', 'Total_Profit']

# Calculate raw percentages first, then round at the end
product_profit['Profit_%']            = (product_profit['Total_Profit'] / total_profit * 100)
product_profit['Cumulative_Profit_%'] = product_profit['Profit_%'].cumsum()

# Round AFTER cumsum
product_profit['Profit_%']            = product_profit['Profit_%'].round(2)
product_profit['Cumulative_Profit_%'] = product_profit['Cumulative_Profit_%'].round(2)

product_profit['Rank'] = range(1, len(product_profit) + 1)

print("=== PRODUCT PARETO — PROFIT ===")
print(product_profit[['Rank', 'Product Name', 'Total_Profit',
                       'Profit_%', 'Cumulative_Profit_%']].to_string(index=False))

=== PRODUCT PARETO — PROFIT ===
 Rank                      Product Name  Total_Profit  Profit_%  Cumulative_Profit_%
    1    Wonka Bar -Scrumdiddlyumptious      11350.00     20.53                20.53
    2 Wonka Bar - Triple Dazzle Caramel      10897.60     19.71                40.24
    3 Wonka Bar - Nutty Crunch Surprise      10311.09     18.65                58.89
    4        Wonka Bar - Milk Chocolate      10062.59     18.20                77.08
    5         Wonka Bar - Fudge Mallows       9732.00     17.60                94.69
    6                Lickable Wallpaper       2480.00      4.49                99.17
    7                         Wonka Gum        170.95      0.31                99.48
    8            Everlasting Gobstopper        104.00      0.19                99.67
    9                       Hair Toffee         59.50      0.11                99.78
   10                         Kazookles         47.50      0.09                99.86
   11              Fizzy Lifting 

### 2. Product Level Pareto (Revenue)

In [4]:
product_revenue = df.groupby('Product Name')['Sales'].sum()\
                    .sort_values(ascending=False)\
                    .reset_index()

product_revenue.columns = ['Product Name', 'Total_Revenue']

# Calculate raw percentages first, then round at the end
product_revenue['Revenue_%']            = (product_revenue['Total_Revenue'] / total_revenue * 100)
product_revenue['Cumulative_Revenue_%'] = product_revenue['Revenue_%'].cumsum()

# Round AFTER cumsum
product_revenue['Revenue_%']            = product_revenue['Revenue_%'].round(2)
product_revenue['Cumulative_Revenue_%'] = product_revenue['Cumulative_Revenue_%'].round(2)

product_revenue['Rank'] = range(1, len(product_revenue) + 1)

print("=== PRODUCT PARETO — REVENUE ===")
print(product_revenue[['Rank', 'Product Name', 'Total_Revenue',
                        'Revenue_%', 'Cumulative_Revenue_%']].to_string(index=False))

=== PRODUCT PARETO — REVENUE ===
 Rank                      Product Name  Total_Revenue  Revenue_%  Cumulative_Revenue_%
    1 Wonka Bar - Triple Dazzle Caramel       16680.00      19.90                 19.90
    2    Wonka Bar -Scrumdiddlyumptious       16344.00      19.50                 39.40
    3        Wonka Bar - Milk Chocolate       15499.25      18.49                 57.88
    4         Wonka Bar - Fudge Mallows       14598.00      17.41                 75.30
    5 Wonka Bar - Nutty Crunch Surprise       14452.09      17.24                 92.54
    6                Lickable Wallpaper        4960.00       5.92                 98.46
    7                         Kazookles         617.50       0.74                 99.19
    8                         Wonka Gum         328.75       0.39                 99.59
    9            Everlasting Gobstopper         130.00       0.16                 99.74
   10                       Hair Toffee          76.50       0.09                 99.83

### 3. Find the 80% Threshold Products

In [5]:
# Products that together make up 80% of profit
profit_80 = product_profit[product_profit['Cumulative_Profit_%'] <= 80]

# If no product lands exactly at 80, include one more to cross it
if profit_80['Cumulative_Profit_%'].max() < 80:
    next_row  = product_profit[product_profit['Cumulative_Profit_%'] > 80].iloc[[0]]
    profit_80 = pd.concat([profit_80, next_row], ignore_index=True)

# Same for revenue
revenue_80 = product_revenue[product_revenue['Cumulative_Revenue_%'] <= 80]
if revenue_80['Cumulative_Revenue_%'].max() < 80:
    next_row   = product_revenue[product_revenue['Cumulative_Revenue_%'] > 80].iloc[[0]]
    revenue_80 = pd.concat([revenue_80, next_row], ignore_index=True)

print("PRODUCTS DRIVING 80% OF PROFIT")
print(f"  Number of products : {len(profit_80)} out of {len(product_profit)}")
print(f"  Products           : {profit_80['Product Name'].tolist()}")
print(f"  Cumulative Profit  : {profit_80['Cumulative_Profit_%'].max():.2f}%")

print("\nPRODUCTS DRIVING 80% OF REVENUE")
print(f"  Number of products : {len(revenue_80)} out of {len(product_revenue)}")
print(f"  Products           : {revenue_80['Product Name'].tolist()}")
print(f"  Cumulative Revenue : {revenue_80['Cumulative_Revenue_%'].max():.2f}%")

PRODUCTS DRIVING 80% OF PROFIT
  Number of products : 5 out of 15
  Products           : ['Wonka Bar -Scrumdiddlyumptious', 'Wonka Bar - Triple Dazzle Caramel', 'Wonka Bar - Nutty Crunch Surprise', 'Wonka Bar - Milk Chocolate', 'Wonka Bar - Fudge Mallows']
  Cumulative Profit  : 94.69%

PRODUCTS DRIVING 80% OF REVENUE
  Number of products : 5 out of 15
  Products           : ['Wonka Bar - Triple Dazzle Caramel', 'Wonka Bar -Scrumdiddlyumptious', 'Wonka Bar - Milk Chocolate', 'Wonka Bar - Fudge Mallows', 'Wonka Bar - Nutty Crunch Surprise']
  Cumulative Revenue : 92.54%


### 4. Pareto by Division

In [6]:
total_profit_div = df['Gross Profit'].sum()

division_pareto = df.groupby('Division')['Gross Profit'].sum()\
                    .sort_values(ascending=False)\
                    .reset_index()

division_pareto.columns = ['Division', 'Total_Profit']
division_pareto['Profit_%']            = (division_pareto['Total_Profit'] / total_profit_div * 100).round(2)
division_pareto['Cumulative_Profit_%'] = division_pareto['Profit_%'].cumsum().round(2)

print("DIVISION PARETO — PROFIT")
print(division_pareto.to_string(index=False))

DIVISION PARETO — PROFIT
 Division  Total_Profit  Profit_%  Cumulative_Profit_%
Chocolate      52353.28     94.69                94.69
    Other       2698.45      4.88                99.57
    Sugar        239.64      0.43               100.00


### 5. Pareto by Factory

In [7]:
factory_pareto = df.groupby('Factory')['Gross Profit'].sum()\
                   .sort_values(ascending=False)\
                   .reset_index()

factory_pareto.columns = ['Factory', 'Total_Profit']
factory_pareto['Profit_%']            = (factory_pareto['Total_Profit'] / total_profit * 100).round(2)
factory_pareto['Cumulative_Profit_%'] = factory_pareto['Profit_%'].cumsum().round(2)

print("FACTORY PARETO — PROFIT")
print(factory_pareto.to_string(index=False))

FACTORY PARETO — PROFIT
          Factory  Total_Profit  Profit_%  Cumulative_Profit_%
    Lot's O' Nuts      31393.09     56.78                56.78
  Wicked Choccy's      20960.19     37.91                94.69
   Secret Factory       2754.95      4.98                99.67
The Other Factory        107.00      0.19                99.86
      Sugar Shack         76.14      0.14               100.00


### 6. Pareto by Region

In [8]:
region_pareto = df.groupby('Region')['Gross Profit'].sum()\
                  .sort_values(ascending=False)\
                  .reset_index()

region_pareto.columns = ['Region', 'Total_Profit']
region_pareto['Profit_%']            = (region_pareto['Total_Profit'] / total_profit * 100).round(2)
region_pareto['Cumulative_Profit_%'] = region_pareto['Profit_%'].cumsum().round(2)

print("REGION PARETO — PROFIT")
print(region_pareto.to_string(index=False))

REGION PARETO — PROFIT
  Region  Total_Profit  Profit_%  Cumulative_Profit_%
 Pacific      18122.34     32.78                32.78
Atlantic      15901.34     28.76                61.54
Interior      12867.99     23.27                84.81
    Gulf       8399.70     15.19               100.00


### 7. Dependency Risk Assessment

In [9]:
top1_profit_share  = product_profit.iloc[0]['Profit_%']
top3_profit_share  = product_profit.iloc[:3]['Profit_%'].sum()
top5_profit_share  = product_profit.iloc[:5]['Profit_%'].sum()
bottom10_share     = product_profit.iloc[5:]['Profit_%'].sum()
top_division_share = division_pareto.iloc[0]['Profit_%']
top_factory_share  = factory_pareto.iloc[0]['Profit_%']

print("=== DEPENDENCY RISK ASSESSMENT ===")
print()
print(f"  Top 1 product  → {top1_profit_share:.1f}% of total profit")
print(f"  Top 3 products → {top3_profit_share:.1f}% of total profit")
print(f"  Top 5 products → {top5_profit_share:.1f}% of total profit")
print(f"  Bottom 10 products → only {bottom10_share:.1f}% of total profit")
print()
print(f"  Top division ({division_pareto.iloc[0]['Division']}) → {top_division_share:.1f}% of profit")
print(f"  Top factory  ({factory_pareto.iloc[0]['Factory']})  → {top_factory_share:.1f}% of profit")
print()

# Risk flags
if top5_profit_share > 90:
    print("CRITICAL: Top 5 products generate over 90% of profit.")
    print("     Loss of any single Wonka Bar variant would severely impact the business.")

if top_division_share > 90:
    print("CRITICAL: Single division dominates profit.")
    print("     Portfolio diversification is urgently needed.")

if top_factory_share > 50:
    print(f"WARNING: Over 50% of profit depends on one factory.")
    print(f"     Any disruption at {factory_pareto.iloc[0]['Factory']} is a major operational risk.")

=== DEPENDENCY RISK ASSESSMENT ===

  Top 1 product  → 20.5% of total profit
  Top 3 products → 58.9% of total profit
  Top 5 products → 94.7% of total profit
  Bottom 10 products → only 5.3% of total profit

  Top division (Chocolate) → 94.7% of profit
  Top factory  (Lot's O' Nuts)  → 56.8% of profit

CRITICAL: Top 5 products generate over 90% of profit.
     Loss of any single Wonka Bar variant would severely impact the business.
CRITICAL: Single division dominates profit.
     Portfolio diversification is urgently needed.
     Any disruption at Lot's O' Nuts is a major operational risk.


### Save Report

In [10]:
pareto_report = {
    "product_pareto_profit"  : product_profit.to_dict(orient='records'),
    "product_pareto_revenue" : product_revenue.to_dict(orient='records'),
    "division_pareto"        : division_pareto.to_dict(orient='records'),
    "factory_pareto"         : factory_pareto.to_dict(orient='records'),
    "region_pareto"          : region_pareto.to_dict(orient='records'),
    "80_percent_threshold": {
        "products_driving_80pct_profit" : profit_80['Product Name'].tolist(),
        "products_driving_80pct_revenue": revenue_80['Product Name'].tolist()
    },
    "dependency_risk": {
        "top_1_product_profit_share_%"  : round(top1_profit_share, 2),
        "top_3_products_profit_share_%" : round(top3_profit_share, 2),
        "top_5_products_profit_share_%" : round(top5_profit_share, 2),
        "bottom_10_products_share_%"    : round(bottom10_share, 2),
        "top_division_profit_share_%"   : round(top_division_share, 2),
        "top_factory_profit_share_%"    : round(top_factory_share, 2),
        "risk_level"                    : "CRITICAL" if top5_profit_share > 90 else "MODERATE"
    }
}

report_path = '../outputs/reports/pareto_analysis_report.json'
os.makedirs(os.path.dirname(report_path), exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(pareto_report, f, indent=4)

print(f"Pareto analysis report saved to: {report_path}")

Pareto analysis report saved to: ../outputs/reports/pareto_analysis_report.json
